# Multimodal AI Integration & Assistant

**Day 1 — Part C & Lab** | Topics from the course outline:
- Text generation and summarization *(primary focus)*
- Image understanding via vision API *(brief — enterprise scenarios only)*
- Speech-to-text and text-to-speech
- Multimodal reasoning workflows
- **Lab:** Build a multimodal assistant (voice/text in → AI → text/speech out)

**What you will build:**
1. Reusable helpers for text summarization, support drafts, and structured JSON output — all via the same `chat/completions` endpoint with different system prompts.
2. A vision helper that describes screenshots / charts.
3. STT and TTS wrappers.
4. An end-to-end pipeline that chains these blocks together.

**Prerequisites:** Completed Part A+B (`01-Python-and-AI-APIs.ipynb`). Python 3.11, `.venv-student`, `.env` with `OPENAI_API_KEY`, `config.json` (see SETUP.md).


## 1. Pipeline architecture — what we build today

```
  ┌───────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
  │  Block 1: STT  │───▶│  Block 2:     │───▶│  Block 3:     │───▶│  Audio Reply   │
  │  Voice → Text  │     │  Chat (LLM)  │     │  TTS          │     │  (.mp3 file)  │
  │  speech_to_    │     │  call_chat_   │     │  text_to_     │     │               │
  │  text()        │     │  api()        │     │  speech()     │     │               │
  └───────────────┘     └──────────────┘     └──────────────┘     └──────────────┘
        ▲                       │                                         │
        │                       ▼                                         │
   Audio file (.mp3)    Optional: Vision                            Play locally
   OR text fallback     (image context)
```

The **same REST pattern** (`requests.post()` → parse response) applies to every modality — only the payload and response format change:

| Modality | API endpoint | Request format | Response format | Enterprise use case |
|----------|-------------|----------------|-----------------|---------------------|
| Text → Text | `/chat/completions` | JSON (messages) | JSON (choices[0].message.content) | Summaries, support drafts, classification |
| Image → Text | `/chat/completions` | JSON (messages + image_url) | JSON (text description) | Screenshot analysis, invoice OCR |
| Audio → Text | `/audio/transcriptions` | Multipart (file + model) | JSON (text) | Meeting notes, voice commands |
| Text → Audio | `/audio/speech` | JSON (input + voice) | Binary audio | Voice assistants, accessibility |

Each section below implements one block. Section 6 chains all blocks into the pipeline.


## 2. Setup — load secrets and config (run first)

Same pattern as Part A: `.env` for secrets, `config.json` for non-secret settings. Run Jupyter from the `hands-on` folder.


In [1]:
import os
import json
import sys
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
with open(Path.cwd() / "config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

API_KEY = os.environ.get("OPENAI_API_KEY", "")
base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
CHAT_URL = f"{base}/chat/completions"
STT_URL = f"{base}/audio/transcriptions"
TTS_URL = f"{base}/audio/speech"
MODEL_CHAT = CONFIG.get("model_chat", "gpt-4o-mini")
MODEL_STT = CONFIG.get("model_stt", "whisper-1")
MODEL_TTS = CONFIG.get("model_tts", "tts-1")
TIMEOUT = CONFIG.get("request_timeout_seconds", 60)

checks = {
    "Python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "API base": base,
    "Chat model": MODEL_CHAT,
    "API key": "set" if API_KEY else "NOT SET — add to .env",
}
for k, v in checks.items():
    print(f"  {k}: {v}")


  Python: 3.11
  API base: https://api.openai.com/v1
  Chat model: gpt-4o-mini
  API key: set


## 3. Text generation and summarization (primary focus)

All text scenarios use the **same `chat/completions` endpoint** — the system prompt determines the behaviour. We define three reusable helpers first, then apply them to enterprise scenarios.


In [2]:
def build_chat_payload(system_content, user_content, max_tokens=300,
                       temperature=0.3, response_format=None):
    """Build OpenAI chat/completions request body."""
    payload = {
        "model": MODEL_CHAT,
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content[:8000]},
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if response_format:
        payload["response_format"] = response_format
    return payload


def call_chat_api(payload, api_key=None):
    """POST to chat/completions; return full response JSON."""
    api_key = api_key or API_KEY
    if not api_key:
        return {"choices": [{"message": {"content": "[Set OPENAI_API_KEY]"}}]}
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    resp = requests.post(CHAT_URL, json=payload, headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def extract_content(resp_json):
    """Extract assistant text from chat/completions response."""
    return resp_json.get("choices", [{}])[0].get("message", {}).get("content", "")


print("Core helpers defined: build_chat_payload, call_chat_api, extract_content")


Core helpers defined: build_chat_payload, call_chat_api, extract_content


### Scenario 1 — Executive summary (internal reports)

**Use case:** Leadership receives long reports. Your pipeline returns concise bullet-point summaries so the CTO can review all regions in 5 minutes.


In [3]:
def summarize_for_executive(long_text, bullet_count=3):
    """Produce an N-bullet executive summary."""
    payload = build_chat_payload(
        f"You write concise executive summaries. Output exactly {bullet_count} bullet points. "
        f"Each bullet starts with '- '.",
        long_text,
        max_tokens=400,
    )
    return extract_content(call_chat_api(payload))


sample_report = (
    "Q3 results show revenue up 12% YoY driven by enterprise contracts in APAC. "
    "Operations expanded to two new regions (LATAM and MEA) with local offices. "
    "Customer satisfaction scores improved from 4.1 to 4.5 out of 5. "
    "Two product launches delayed to Q4 due to auth platform dependency. "
    "Engineering headcount grew by 15%. Cloud costs up 8% but unit cost per customer down 3%. "
    "Churn rate decreased from 5.2% to 4.1% — lowest in company history."
)

print("Executive summary:")
print(summarize_for_executive(sample_report))


Executive summary:
- Revenue increased by 12% YoY, primarily due to enterprise contracts in the APAC region.  
- Customer satisfaction improved significantly, rising from 4.1 to 4.5 out of 5, while churn rate hit a record low of 4.1%.  
- Operations expanded into LATAM and MEA with local offices, although two product launches were delayed to Q4 due to dependencies.  


### Scenario 2 — First-draft support reply

**Use case:** Support agents receive 200+ tickets/day. Your pipeline generates a draft from the ticket text plus a knowledge-base snippet. The agent reviews, edits, and sends — reducing reply time from 8 min to 2 min.

**Human-in-the-loop:** The model drafts; a human reviews before sending.


In [4]:
def draft_support_reply(ticket_text, knowledge_snippet):
    """Generate a short, professional reply using ticket + knowledge."""
    user_content = f"Knowledge:\n{knowledge_snippet}\n\nTicket:\n{ticket_text}"
    payload = build_chat_payload(
        "You write brief, professional support replies. "
        "Use only the provided knowledge. If the knowledge does not cover the issue, "
        "say so and suggest contacting support.",
        user_content,
        max_tokens=200,
    )
    return extract_content(call_chat_api(payload))


ticket = "Customer cannot reset password; getting error 500 every time they click the link."
knowledge = (
    "Password reset: use the Forgot Password link on the login page. "
    "If error 500, clear browser cache or try incognito mode. "
    "If the issue persists, escalate to engineering with the request ID."
)
print("Draft reply:")
print(draft_support_reply(ticket, knowledge))


Draft reply:
Please advise the customer to clear their browser cache or try using incognito mode to reset their password. If the error 500 persists, please escalate the issue to engineering with the request ID.


### Scenario 3 — Structured JSON output (risk classification)

**Use case:** Compliance pipeline classifies reports into risk levels. Downstream dashboards consume structured JSON — not free text. The `response_format` parameter forces valid JSON output.


In [5]:
def classify_risk_as_json(text):
    """Classify text into risk_level with structured JSON output."""
    system = (
        "You are a risk classification assistant. Respond ONLY in JSON with keys: "
        '{"risk_level": "low|medium|high|critical", '
        '"reasons": ["reason1", "reason2"], '
        '"recommended_action": "one sentence"}.'
    )
    payload = build_chat_payload(
        system, text, max_tokens=300, temperature=0.1,
        response_format={"type": "json_object"},
    )
    raw = extract_content(call_chat_api(payload))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_response": raw, "parse_error": True}


incident_report = (
    "Production database experienced 45 minutes of downtime during peak hours. "
    "Approximately 12,000 customers were affected. Root cause: expired TLS certificate. "
    "Failover did not trigger due to misconfigured health checks. No data loss confirmed."
)
print("Risk classification:")
print(json.dumps(classify_risk_as_json(incident_report), indent=2))


Risk classification:
{
  "risk_level": "high",
  "reasons": [
    "Significant downtime during peak hours affecting a large number of customers.",
    "Expired TLS certificate indicates a lapse in security management.",
    "Misconfigured health checks prevented failover, increasing the risk of future incidents."
  ],
  "recommended_action": "Immediate review and update of certificate management processes and health check configurations is essential."
}


### Prompt engineering — Audience-specific system prompts

**Pattern:** Same input, different system prompts → different outputs. One API, three prompts for executive, engineering, and customer audiences.


In [6]:
def summarize_for_audience(text, audience="executive"):
    """Summarize with audience-specific system prompt."""
    prompts = {
        "executive": "Summarize for a C-level executive. 3 bullet points. Focus on revenue, risk, decisions.",
        "engineering": "Summarize for a senior engineer. 3 bullet points. Focus on technical details, root cause.",
        "customer": "Summarize for a customer communication. 2-3 friendly sentences. Non-technical language.",
    }
    payload = build_chat_payload(prompts.get(audience, prompts["executive"]), text, max_tokens=300)
    return extract_content(call_chat_api(payload))


incident_text = (
    "At 14:32 UTC, the primary database cluster failed over due to an expired TLS certificate. "
    "Failover took 45 minutes; 12,000 customers experienced errors. Revenue impact: ~$18,000. "
    "Root cause: certificate not in automated rotation. Fix: added to rotation; health checks updated."
)
for audience in ["executive", "engineering", "customer"]:
    print(f"\n--- {audience.upper()} ---")
    print(summarize_for_audience(incident_text, audience))



--- EXECUTIVE ---
- **Revenue Impact**: The database cluster failure resulted in an estimated revenue loss of approximately $18,000 due to service disruptions affecting 12,000 customers.  
- **Risk Mitigation**: The incident highlighted a critical risk related to certificate management, leading to the implementation of automated rotation processes to prevent future occurrences.  
- **Decision Making**: Immediate corrective actions were taken, including updating health checks and integrating the expired certificate into the automated rotation, ensuring enhanced operational resilience moving forward.  

--- ENGINEERING ---
- The primary database cluster experienced a failover at 14:32 UTC due to an expired TLS certificate, resulting in a 45-minute downtime that affected 12,000 customers.
- The root cause was identified as the TLS certificate not being included in the automated rotation process, leading to the expiration without timely renewal.
- To mitigate future risks, the certificate

## 4. Vision API — image to text (brief)

**Enterprise scenarios:** support screenshot analysis, accessibility alt-text, invoice OCR. Same REST pattern — the `messages` array includes an `image_url` item alongside text.

> Kept brief per course feedback. The primary Day 1 focus is text and speech.


In [8]:
def chat_with_image(user_text, image_url, system_prompt=None):
    """Send text + image to vision-capable chat model; return text description."""
    if not API_KEY:
        return None
    system_prompt = system_prompt or "Describe what you see and answer the question."
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    body = {
        "model": MODEL_CHAT,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "text", "text": user_text},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]},
        ],
        "max_tokens": 300,
    }
    resp = requests.post(CHAT_URL, json=body, headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


# Example (uncomment with a real image URL):
print(chat_with_image("What error does this screenshot show?", "https://thelogocompany.net/wp-content/uploads/2023/04/red.png"))
print("chat_with_image(text, image_url) defined.")
print("Same POST + JSON; content list includes {type: image_url}.")


The screenshot appears to be a movie poster for "Home Again," featuring Reese Witherspoon. The text on the poster states, "Starting over is not for beginners." There is no immediate visual error shown in the design or layout of the poster from what can be seen. If you're looking for a specific type of error (such as a typo, factual inaccuracy, or design flaw), please let me know!
chat_with_image(text, image_url) defined.
Same POST + JSON; content list includes {type: image_url}.


## 5. Speech APIs — STT and TTS

- **STT (Block 1):** POST audio file as multipart form data → JSON with `text` field.
- **TTS (Block 3):** POST JSON with text → binary audio bytes (save to file).

These implement the first and last blocks of the assistant pipeline.


In [9]:
SUPPORTED_AUDIO = {".mp3", ".mp4", ".mpeg", ".mpga", ".m4a", ".wav", ".webm"}
TTS_VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]


def speech_to_text(audio_path):
    """Block 1: POST audio file to STT API; return transcript."""
    if not API_KEY:
        return None
    path = Path(audio_path)
    if not path.exists():
        print(f"File not found: {audio_path}")
        return None
    if path.suffix.lower() not in SUPPORTED_AUDIO:
        print(f"Unsupported format: {path.suffix}. Supported: {SUPPORTED_AUDIO}")
        return None
    headers = {"Authorization": f"Bearer {API_KEY}"}
    with open(audio_path, "rb") as f:
        resp = requests.post(
            STT_URL, files={"file": (path.name, f, "audio/mpeg")},
            data={"model": MODEL_STT}, headers=headers, timeout=120,
        )
    resp.raise_for_status()
    return resp.json().get("text", "")


def text_to_speech(text, output_path, voice="alloy", speed=1.0):
    """Block 3: POST text to TTS API; save audio file."""
    if not API_KEY:
        return False
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    body = {"model": MODEL_TTS, "input": text, "voice": voice, "speed": speed}
    resp = requests.post(TTS_URL, json=body, headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    Path(output_path).write_bytes(resp.content)
    return True


print(f"speech_to_text(path)  — endpoint: {STT_URL}, model: {MODEL_STT}")
print(f"text_to_speech(text, path, voice, speed)  — endpoint: {TTS_URL}, model: {MODEL_TTS}")
print(f"Voices: {TTS_VOICES}")


speech_to_text(path)  — endpoint: https://api.openai.com/v1/audio/transcriptions, model: whisper-1
text_to_speech(text, path, voice, speed)  — endpoint: https://api.openai.com/v1/audio/speech, model: tts-1
Voices: ['alloy', 'echo', 'fable', 'onyx', 'nova', 'shimmer']


## 6. End-to-end pipeline: Block 1 → 2 → 3

```
  ┌──────────┐      ┌──────────┐      ┌──────────┐
  │ Block 1  │─────▶│ Block 2  │─────▶│ Block 3  │
  │ STT      │      │ Chat     │      │ TTS      │
  │ (skip if │      │ (always) │      │ (optional)│
  │ text)    │      │          │      │          │
  └──────────┘      └──────────┘      └──────────┘
```

**Fallback path:** No audio file? Start with text input (skip Block 1). No TTS needed? Omit `tts_path` (skip Block 3). The chat block always runs.

**Error recovery:** If TTS fails, the text answer is still returned. If STT fails, fall back to text input.


In [10]:
def run_assistant(audio_path=None, text_input=None,
                  system_prompt="You are a helpful assistant. Answer concisely.",
                  tts_path=None, tts_voice="alloy"):
    """Run pipeline: (Block 1) STT or text → (Block 2) Chat → (Block 3) optional TTS.
    Returns dict with user_text, assistant_text, tts_path, timings, errors."""
    result = {"user_text": None, "assistant_text": None, "tts_path": None,
              "timings": {}, "errors": []}

    # Block 1: STT (or text fallback)
    if audio_path and Path(audio_path).is_file():
        t0 = time.time()
        try:
            result["user_text"] = speech_to_text(audio_path)
        except Exception as e:
            result["errors"].append(f"STT failed: {e}")
        result["timings"]["stt_s"] = round(time.time() - t0, 2)
    if not result["user_text"] and text_input:
        result["user_text"] = text_input
    if not result["user_text"]:
        result["errors"].append("No audio or text provided")
        return result

    # Block 2: Chat (always runs)
    t0 = time.time()
    try:
        payload = build_chat_payload(system_prompt, result["user_text"], max_tokens=300)
        result["assistant_text"] = extract_content(call_chat_api(payload))
    except Exception as e:
        result["errors"].append(f"Chat failed: {e}")
    result["timings"]["chat_s"] = round(time.time() - t0, 2)
    if not result["assistant_text"]:
        return result

    # Block 3: TTS (optional — skip on failure)
    if tts_path and API_KEY:
        t0 = time.time()
        try:
            text_to_speech(result["assistant_text"], tts_path, voice=tts_voice)
            result["tts_path"] = tts_path
        except Exception as e:
            result["errors"].append(f"TTS failed (text still available): {e}")
        result["timings"]["tts_s"] = round(time.time() - t0, 2)

    return result


def print_result(r):
    """Pretty-print pipeline result."""
    print(f"  User:      {r['user_text']}")
    print(f"  Assistant: {r['assistant_text']}")
    if r["tts_path"]:
        print(f"  Audio:     {r['tts_path']}")
    if r["timings"]:
        print(f"  Timings:   {r['timings']}")
    if r["errors"]:
        print(f"  Warnings:  {r['errors']}")


### Variant A — Text only (fastest path)

Start here to verify Block 2 works.


In [11]:
result_a = run_assistant(text_input="Explain REST APIs in two sentences.")
print("=== Variant A: Text only ===")
print_result(result_a)


=== Variant A: Text only ===
  User:      Explain REST APIs in two sentences.
  Assistant: REST APIs (Representational State Transfer Application Programming Interfaces) are a set of rules that allow different software applications to communicate over the internet using standard HTTP methods like GET, POST, PUT, and DELETE. They enable the exchange of data in a stateless manner, typically using JSON or XML formats, making it easier for developers to create scalable and interoperable web services.
  Timings:   {'chat_s': 2.3}


### Variant B — Text + TTS (audio output)

Same as A, plus Block 3 saves an audio file.


In [12]:
result_b = run_assistant(
    text_input="Give three benefits of multimodal AI assistants.",
    tts_path="assistant_reply.mp3",
    tts_voice="nova",
)
print("=== Variant B: Text + TTS ===")
print_result(result_b)


=== Variant B: Text + TTS ===
  User:      Give three benefits of multimodal AI assistants.
  Assistant: 1. **Enhanced User Experience**: Multimodal AI assistants can process and respond to inputs in various forms (text, voice, images), providing a more intuitive and engaging interaction for users.

2. **Improved Accessibility**: They cater to diverse user needs and preferences, allowing individuals with different abilities or communication styles to interact effectively.

3. **Comprehensive Information Processing**: By integrating multiple modalities, these assistants can analyze and synthesize information more holistically, leading to more accurate and context-aware responses.
  Audio:     assistant_reply.mp3
  Timings:   {'chat_s': 2.95, 'tts_s': 10.57}


### System prompt variants — same question, different behaviour

The same user input produces very different outputs depending on the system prompt. Try all three and compare.


In [14]:
SYSTEM_PROMPTS = {
    "general": "You are a helpful assistant. Answer concisely and accurately.",
    "technical": (
        "You are a senior software engineer. Give technically precise answers. "
        "Include trade-offs and best practices."
    ),
    "support": (
        "You are a customer support agent. Be friendly and actionable. "
        "Give step-by-step instructions in simple language."
    ),
}

question = "How do I handle API rate limits in my application?"
for variant, prompt in SYSTEM_PROMPTS.items():
    r = run_assistant(text_input=question, system_prompt=prompt)
    print(f"\n--- {variant.upper()} ---")
    print(r["assistant_text"][:250] if r["assistant_text"] else "No response")



--- GENERAL ---
To handle API rate limits in your application, follow these strategies:

1. **Understand Rate Limits**: Check the API documentation for specific rate limits (requests per second/minute/hour).

2. **Implement Backoff Strategies**: Use exponential back

--- TECHNICAL ---
Handling API rate limits is crucial for ensuring that your application remains functional and compliant with the API provider's usage policies. Here are some best practices and trade-offs to consider when implementing rate limit handling in your appl

--- SUPPORT ---
Handling API rate limits is important to ensure your application runs smoothly without interruptions. Here’s a step-by-step guide to help you manage API rate limits effectively:

### Step 1: Understand the Rate Limits
- **Check Documentation**: Start


## 7. Frameworks cheat sheet

| Library | Used for | Key functions / classes |
|---------|----------|------------------------|
| `requests` | HTTP calls to AI APIs | `requests.post(url, json=..., headers=..., timeout=...)` |
| `python-dotenv` | Load secrets from `.env` | `load_dotenv(path)`, then `os.environ.get("KEY")` |
| `json` | Parse / build payloads | `json.loads(s)`, `json.dumps(obj, indent=2)` |
| `pathlib` | File paths | `Path(f).exists()`, `Path(f).read_bytes()`, `Path(f).write_bytes(b)` |
| `logging` | Structured logs | `logging.getLogger(name)`, `logger.info(msg)` |
| `csv` | Read/write CSV data | `csv.DictReader(f)`, `csv.DictWriter(f, fieldnames=[...])` |
| `time` | Measure latency | `time.time()` for start/end timing |
| `base64` | Encode images for vision API | `base64.standard_b64encode(raw).decode("ascii")` |
| `re` | Post-process model output | `re.sub(pattern, repl, text)` |

> These are the same libraries used throughout the course. Day 2 adds `transformers`, `sentence-transformers`, and `faiss`.
